# Task 4: Disease Prediction from Medical Data
**Dataset:** Breast Cancer Wisconsin (Diagnostic) – UCI ML Repository (bundled with scikit-learn)  
**Models:** Logistic Regression, SVM, Random Forest, XGBoost  
Run all cells in order (Runtime → Run all).

In [ ]:
!pip install -q xgboost scikit-learn pandas matplotlib

## 1. Load data

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier

SEED = 42
data = load_breast_cancer(as_frame=True)
X, y = data.data, 1 - data.target   # 1 = malignant (disease), 0 = benign (healthy)

print(f"Samples: {X.shape[0]}, Features: {X.shape[1]}")
print(f"Disease (malignant) = {y.sum()} ({y.mean():.1%}), Healthy (benign) = {(1-y).sum()}")
print("Missing values:", int(X.isna().sum().sum()))
X.head()

## 2. Exploratory analysis

In [ ]:
corr_target = X.corrwith(y).sort_values(ascending=False)
print(corr_target.head(5).round(3))

fig, ax = plt.subplots(figsize=(7, 6))
corr_target.plot.barh(ax=ax, color=np.where(corr_target > 0, "#c0392b", "#2980b9"))
ax.set_title("Correlation of each feature with malignancy"); ax.tick_params(labelsize=7)
plt.tight_layout(); plt.show()

## 3. Train/test split (stratified 80/20)

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print(X_tr.shape, X_te.shape)

## 4. Train and tune the four models (5-fold grid search on AUC)

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
models = {
    "Logistic Regression": (Pipeline([("sc", StandardScaler()), ("m", LogisticRegression(max_iter=5000))]),
                            {"m__C": [0.01, 0.1, 1, 10, 100]}),
    "SVM": (Pipeline([("sc", StandardScaler()), ("m", SVC(probability=True, random_state=SEED))]),
            {"m__C": [0.1, 1, 10], "m__gamma": ["scale", 0.01, 0.1], "m__kernel": ["rbf", "linear"]}),
    "Random Forest": (RandomForestClassifier(random_state=SEED),
                      {"n_estimators": [200, 400], "max_depth": [None, 6, 10], "min_samples_leaf": [1, 3]}),
    "XGBoost": (XGBClassifier(eval_metric="logloss", random_state=SEED),
                {"n_estimators": [100, 300], "max_depth": [2, 3, 5], "learning_rate": [0.03, 0.1]}),
}

rows, fitted, probs = [], {}, {}
for name, (est, grid) in models.items():
    gs = GridSearchCV(est, grid, cv=cv, scoring="roc_auc", n_jobs=-1).fit(X_tr, y_tr)
    best = gs.best_estimator_
    pred, prob = best.predict(X_te), best.predict_proba(X_te)[:, 1]
    fitted[name], probs[name] = best, prob
    rows.append({"Model": name, "CV AUC": gs.best_score_,
                 "Accuracy": accuracy_score(y_te, pred), "Precision": precision_score(y_te, pred),
                 "Recall": recall_score(y_te, pred), "F1": f1_score(y_te, pred),
                 "Test AUC": roc_auc_score(y_te, prob)})
    print(f"{name}: best params = {gs.best_params_}")

res = pd.DataFrame(rows).set_index("Model").round(4)
res

## 5. ROC curves and confusion matrices

In [ ]:
plt.figure(figsize=(6, 5))
for n, p in probs.items():
    fpr, tpr, _ = roc_curve(y_te, p)
    plt.plot(fpr, tpr, label=f"{n} (AUC={roc_auc_score(y_te, p):.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=0.8)
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("ROC curves – test set"); plt.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

fig, axs = plt.subplots(1, 4, figsize=(14, 3.4))
for ax, (n, m) in zip(axs, fitted.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_te, m.predict(X_te)),
                           display_labels=["Healthy", "Disease"]).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(n, fontsize=10)
plt.tight_layout(); plt.show()

## 6. Interpretability – which features matter?

In [ ]:
best_name = res["Recall"].add(res["Test AUC"]).idxmax()
pi = permutation_importance(fitted[best_name], X_te, y_te, scoring="roc_auc",
                            n_repeats=30, random_state=SEED, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values().tail(10)
print("Best model:", best_name)
imp.plot.barh(color="#27ae60", figsize=(6, 4))
plt.title(f"Top 10 features – permutation importance ({best_name})"); plt.xlabel("Drop in AUC")
plt.tight_layout(); plt.show()

## 7. Robustness check (5-fold CV on the full data)

In [ ]:
for n, m in fitted.items():
    s = cross_val_score(m, X, y, cv=cv, scoring="accuracy")
    print(f"{n:20s} {s.mean():.4f} ± {s.std():.4f}")

## 8. Conclusions
- SVM (RBF) gave the best overall test performance; Logistic Regression was nearly as good, so a simple interpretable model is competitive here.
- In screening, missed cancers (false negatives) are the costly error, so in practice the decision threshold should be lowered to raise recall.
- Limitations: small test set (114 patients), single research dataset, no external validation.
- To extend to Heart Disease / Diabetes: swap the loader, add imputation + one-hot encoding, and handle class imbalance (`class_weight` or SMOTE).